# ML Mathematics Playground: 03. Probability & Statistics

AI systems are probabilistic at heart. They predict probabilities of categories (e.g., Cat vs Dog) or estimate uncertainty.

In this notebook, we will explore:
1. **Gaussian/Normal Distribution**: The core probability distribution in nature.
2. **Bayes' Theorem & Classifier**: Making decisions using likelihoods and prior beliefs.


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import ipywidgets as widgets
from ipywidgets import interact, interactive

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (8, 5)


## Section 1: The Gaussian Distribution Playground

The Normal Distribution is defined by:
$$P(x) = \frac{1}{\sigma \sqrt{2\pi}} e^{-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2}$$
where $\mu$ is the **mean** (center) and $\sigma$ is the **standard deviation** (width/spread).

Use the sliders to adjust $\mu$ and $\sigma$ and see how the probability curve updates.


In [ ]:
def plot_gaussian(mu, sigma):
    x = np.linspace(-10, 10, 500)
    y = stats.norm.pdf(x, mu, sigma)
    
    fig, ax = plt.subplots()
    ax.plot(x, y, color='blue', lw=2, label=f'N(μ={mu}, σ={sigma})')
    ax.fill_between(x, y, color='blue', alpha=0.15)
    
    # Annotate standard deviations
    ax.axvline(mu, color='red', linestyle='--', alpha=0.7, label='Mean (μ)')
    ax.axvline(mu + sigma, color='orange', linestyle=':', alpha=0.7, label='μ ± σ')
    ax.axvline(mu - sigma, color='orange', linestyle=':', alpha=0.7)
    
    ax.set_ylim(0, 1.0)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper right')
    ax.set_title('Gaussian Probability Density Function')
    plt.show()

interact(plot_gaussian,
         mu=widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description='Mean (μ)'),
         sigma=widgets.FloatSlider(value=1.5, min=0.2, max=4.0, step=0.1, description='Std Dev (σ)'));


## Section 2: Bayes' Theorem & Classifier

Bayes' Theorem updates the probability of a hypothesis (class $C$) given evidence (feature $x$):
$$P(C|x) = \frac{P(x|C) \cdot P(C)}{P(x)}$$
- $P(C|x)$ is the **Posterior** probability.
- $P(x|C)$ is the **Likelihood** of the feature given the class (often modeled as a Gaussian).
- $P(C)$ is the **Prior** (probability of the class before seeing any data).

Let's build a binary classifier to classify pets as **Cats** or **Dogs** based on their weight.
- Cats: Mean weight $\approx 4$ kg
- Dogs: Mean weight $\approx 12$ kg

Adjust the **Prior Probability of Dog** $P(\text{Dog})$ to see how our classification boundary moves. If dogs are extremely rare (low prior), the classifier requires much stronger weight evidence to classify a pet as a dog!


In [ ]:
def plot_bayes_classifier(prior_dog, cat_mu, cat_sigma, dog_mu, dog_sigma):
    prior_cat = 1.0 - prior_dog
    
    weights = np.linspace(0, 20, 500)
    
    # Likelihoods P(weight | Class)
    likelihood_cat = stats.norm.pdf(weights, cat_mu, cat_sigma)
    likelihood_dog = stats.norm.pdf(weights, dog_mu, dog_sigma)
    
    # Unnormalized Posteriors: Likelihood * Prior
    post_cat = likelihood_cat * prior_cat
    post_dog = likelihood_dog * prior_dog
    
    # Total probability P(weight)
    evidence = post_cat + post_dog
    
    # Normalized Posteriors
    posterior_cat = post_cat / (evidence + 1e-9)
    posterior_dog = post_dog / (evidence + 1e-9)
    
    # Find decision boundary (where P(Dog|x) = P(Cat|x) = 0.5)
    diff = np.abs(posterior_dog - 0.5)
    boundary_idx = np.argmin(diff)
    boundary_val = weights[boundary_idx]
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8), sharex=True)
    
    # Plot Likelihoods
    ax1.plot(weights, likelihood_cat, color='green', lw=2, label='Cat Likelihood P(weight|Cat)')
    ax1.plot(weights, likelihood_dog, color='blue', lw=2, label='Dog Likelihood P(weight|Dog)')
    ax1.fill_between(weights, likelihood_cat, color='green', alpha=0.1)
    ax1.fill_between(weights, likelihood_dog, color='blue', alpha=0.1)
    ax1.set_ylabel('Probability Density')
    ax1.set_title('Likelihoods (Class-Conditional Distributions)')
    ax1.legend(loc='upper right')
    
    # Plot Posteriors
    ax2.plot(weights, posterior_cat, color='green', lw=2, label='Posterior P(Cat|weight)')
    ax2.plot(weights, posterior_dog, color='blue', lw=2, label='Posterior P(Dog|weight)')
    ax2.axvline(boundary_val, color='red', linestyle='--', lw=2, label=f'Decision Boundary: {boundary_val:.2f} kg')
    ax2.set_xlabel('Weight (kg)')
    ax2.set_ylabel('Posterior Probability')
    ax2.set_title('Posteriors (Calculated via Bayes Rule)')
    ax2.legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

interact(plot_bayes_classifier,
         prior_dog=widgets.FloatSlider(value=0.5, min=0.01, max=0.99, step=0.05, description='P(Dog) Prior'),
         cat_mu=widgets.FloatSlider(value=4.0, min=2.0, max=8.0, step=0.5, description='Cat Mean Weight'),
         cat_sigma=widgets.FloatSlider(value=1.5, min=0.5, max=3.0, step=0.1, description='Cat Std Dev'),
         dog_mu=widgets.FloatSlider(value=12.0, min=8.0, max=16.0, step=0.5, description='Dog Mean Weight'),
         dog_sigma=widgets.FloatSlider(value=2.5, min=0.5, max=4.0, step=0.1, description='Dog Std Dev'));
